# Module 4, Notebook 1: EM Algorithm for Dynamic Factor Models

## Learning Objectives

By completing this notebook, you will:

1. **Understand** the EM algorithm's E-step and M-step for DFMs
2. **Implement** Kalman smoother for computing sufficient statistics
3. **Derive** closed-form M-step parameter updates
4. **Monitor** log-likelihood convergence
5. **Compare** EM estimates to PCA-based estimates
6. **Apply** identification constraints in the likelihood framework

## Prerequisites

- Module 2: Kalman filter and smoother
- Module 3: PCA-based estimation
- State-space representation
- Matrix calculus basics

## Estimated Time: 100 minutes

---

## Setup and Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.linalg import solve, cholesky, lstsq
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Set random seed
np.random.seed(42)

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

print("Setup complete!")
print(f"NumPy version: {np.__version__}")

## 1. Conceptual Foundation

### The EM Algorithm

**Expectation-Maximization (EM)** is an iterative algorithm for maximum likelihood estimation when data is incomplete or contains latent variables.

### Why EM for Factor Models?

In a DFM, the factors $F_t$ are **latent** (unobserved). Direct maximization of the likelihood
$$L(\theta) = p(X_{1:T} | \theta)$$
is difficult because it requires integrating out the factors.

**EM's insight:** Treat factors as "missing data". If we knew the factors, ML estimation would be trivial (just regression). So we iterate:

1. **E-step:** Given current parameters $\theta^{(k)}$, compute expected sufficient statistics using factor posteriors
2. **M-step:** Update parameters $\theta^{(k+1)}$ as if expected statistics were observed

### State-Space Form

We write the DFM in state-space form:

**Measurement equation:**
$$X_t = Z \alpha_t + \varepsilon_t, \quad \varepsilon_t \sim N(0, H)$$

**Transition equation:**
$$\alpha_t = T \alpha_{t-1} + R \eta_t, \quad \eta_t \sim N(0, Q)$$

where:
- $\alpha_t = [F_t', F_{t-1}', ..., F_{t-p+1}']'$ is the state vector (stacks current and lagged factors)
- $Z = [\Lambda, 0, ..., 0]$ maps state to observations
- $T$ is the companion-form transition matrix
- $H = \text{diag}(\sigma_{e_1}^2, ..., \sigma_{e_N}^2)$ is measurement error covariance
- $Q$ is factor innovation covariance

### Key Formulas

**E-step:** Run Kalman smoother to compute
- $\hat{\alpha}_{t|T} = E[\alpha_t | X_{1:T}, \theta^{(k)}]$
- $P_{t|T} = \text{Var}[\alpha_t | X_{1:T}, \theta^{(k)}]$
- $P_{t,t-1|T} = \text{Cov}[\alpha_t, \alpha_{t-1} | X_{1:T}, \theta^{(k)}]$

**M-step:** Update parameters via
$$\theta^{(k+1)} = \arg\max_\theta E[\log p(X, \alpha | \theta) | X, \theta^{(k)}]$$

This has closed-form solutions!

## 2. Kalman Smoother Implementation

The E-step requires smoothed factor estimates. We implement the Kalman smoother.

In [ ]:
def kalman_filter(X, Z, H, T_mat, R, Q, alpha_0, P_0):
    """
    Kalman filter forward pass.
    
    Parameters
    ----------
    X : ndarray (T, N)
        Observations
    Z : ndarray (N, m)
        Measurement matrix
    H : ndarray (N, N)
        Measurement error covariance
    T_mat : ndarray (m, m)
        Transition matrix
    R : ndarray (m, r)
        State innovation selector
    Q : ndarray (r, r)
        State innovation covariance
    alpha_0 : ndarray (m,)
        Initial state mean
    P_0 : ndarray (m, m)
        Initial state covariance
        
    Returns
    -------
    alpha_pred : ndarray (T+1, m)
        Predicted states
    alpha_filt : ndarray (T, m)
        Filtered states
    P_pred : ndarray (T+1, m, m)
        Predicted state covariances
    P_filt : ndarray (T, m, m)
        Filtered state covariances
    loglik : float
        Log-likelihood
    """
    T_obs, N = X.shape
    m = Z.shape[1]
    
    # Storage
    alpha_pred = np.zeros((T_obs + 1, m))
    alpha_filt = np.zeros((T_obs, m))
    P_pred = np.zeros((T_obs + 1, m, m))
    P_filt = np.zeros((T_obs, m, m))
    
    # Initialize
    alpha_pred[0] = alpha_0
    P_pred[0] = P_0
    
    loglik = 0.0
    
    for t in range(T_obs):
        # Prediction error
        v_t = X[t] - Z @ alpha_pred[t]
        
        # Prediction error variance
        F_t = Z @ P_pred[t] @ Z.T + H
        F_t = (F_t + F_t.T) / 2  # Ensure symmetry
        
        # Kalman gain
        try:
            K_t = P_pred[t] @ Z.T @ solve(F_t, np.eye(N), assume_a='pos')
        except:
            K_t = np.zeros((m, N))
        
        # Update
        alpha_filt[t] = alpha_pred[t] + K_t @ v_t
        P_filt[t] = P_pred[t] - K_t @ F_t @ K_t.T
        P_filt[t] = (P_filt[t] + P_filt[t].T) / 2
        
        # Predict next
        if t < T_obs - 1:
            alpha_pred[t + 1] = T_mat @ alpha_filt[t]
            P_pred[t + 1] = T_mat @ P_filt[t] @ T_mat.T + R @ Q @ R.T
            P_pred[t + 1] = (P_pred[t + 1] + P_pred[t + 1].T) / 2
        
        # Log-likelihood contribution
        try:
            L = cholesky(F_t, lower=True)
            log_det = 2 * np.sum(np.log(np.diag(L)))
            w = solve(L, v_t, lower=True)
            quad_form = w @ w
            loglik += -0.5 * (log_det + quad_form + N * np.log(2 * np.pi))
        except:
            pass
    
    return alpha_pred, alpha_filt, P_pred, P_filt, loglik


def kalman_smoother(alpha_pred, alpha_filt, P_pred, P_filt, T_mat):
    """
    Kalman smoother backward pass.
    
    Parameters
    ----------
    alpha_pred, alpha_filt : ndarray
        Outputs from kalman_filter
    P_pred, P_filt : ndarray
        Outputs from kalman_filter
    T_mat : ndarray (m, m)
        Transition matrix
        
    Returns
    -------
    alpha_smooth : ndarray (T, m)
        Smoothed states
    P_smooth : ndarray (T, m, m)
        Smoothed state covariances
    P_smooth_lag : ndarray (T, m, m)
        Smoothed lag-one covariances
    """
    T_obs = alpha_filt.shape[0]
    m = alpha_filt.shape[1]
    
    # Storage
    alpha_smooth = np.zeros((T_obs, m))
    P_smooth = np.zeros((T_obs, m, m))
    P_smooth_lag = np.zeros((T_obs, m, m))
    
    # Initialize at T
    alpha_smooth[-1] = alpha_filt[-1]
    P_smooth[-1] = P_filt[-1]
    
    # Backward recursion
    for t in range(T_obs - 2, -1, -1):
        # Smoother gain
        try:
            J_t = P_filt[t] @ T_mat.T @ solve(P_pred[t + 1], np.eye(m))
        except:
            J_t = np.zeros((m, m))
        
        # Smooth state
        alpha_smooth[t] = alpha_filt[t] + J_t @ (alpha_smooth[t + 1] - alpha_pred[t + 1])
        
        # Smooth covariance
        P_smooth[t] = P_filt[t] + J_t @ (P_smooth[t + 1] - P_pred[t + 1]) @ J_t.T
        P_smooth[t] = (P_smooth[t] + P_smooth[t].T) / 2
        
        # Lag-one covariance
        P_smooth_lag[t + 1] = J_t @ P_smooth[t + 1]
    
    return alpha_smooth, P_smooth, P_smooth_lag


print("Kalman filter and smoother implemented!")

## 3. EM Algorithm: Complete Implementation

Now we implement the full EM algorithm with E-step and M-step.

In [ ]:
class EMDynamicFactorModel:
    """
    EM algorithm for Dynamic Factor Model estimation.
    
    Model:
        X_t = Lambda * F_t + e_t,  e_t ~ N(0, Sigma_e)
        F_t = Phi_1 * F_{t-1} + ... + Phi_p * F_{t-p} + eta_t,  eta_t ~ N(0, Q)
    
    Parameters
    ----------
    n_factors : int
        Number of factors
    n_lags : int, default=1
        VAR lag order for factors
    """
    
    def __init__(self, n_factors, n_lags=1):
        self.r = n_factors
        self.p = n_lags
        self.m = n_factors * n_lags  # State dimension
        
        # Parameters (to be estimated)
        self.Z = None
        self.H = None
        self.T_mat = None
        self.R = None
        self.Q = None
        
        # Results
        self.loglik_hist_ = []
        self.n_iter_ = 0
        
    def initialize_from_pca(self, X):
        """
        Initialize parameters using PCA.
        
        This provides a good starting point close to the MLE.
        """
        T_obs, N = X.shape
        
        # Standardize
        X_mean = np.mean(X, axis=0)
        X_std = np.std(X, axis=0, ddof=1)
        X_std[X_std < 1e-10] = 1.0
        X_proc = (X - X_mean) / X_std
        
        # PCA for initial factors and loadings
        from sklearn.decomposition import PCA
        pca = PCA(n_components=self.r)
        F_init = pca.fit_transform(X_proc)
        Lambda_init = pca.components_.T  # N x r
        
        # Enforce identification: lower triangular for first r rows
        Lambda_init = self._enforce_identification(Lambda_init)
        
        # VAR for factor dynamics
        from statsmodels.tsa.api import VAR
        try:
            var_model = VAR(F_init)
            var_result = var_model.fit(self.p, trend='nc')
            
            # Extract Phi matrices
            Phi_list = []
            for lag in range(1, self.p + 1):
                start_idx = (lag - 1) * self.r
                end_idx = lag * self.r
                Phi_list.append(var_result.params[start_idx:end_idx, :].T)
            
            Q_init = np.cov(var_result.resid, rowvar=False)
        except:
            # Fallback: simple AR(1) with diagonal Phi
            Phi_list = [np.eye(self.r) * 0.5]
            if self.p > 1:
                Phi_list.extend([np.zeros((self.r, self.r))] * (self.p - 1))
            Q_init = np.eye(self.r)
        
        # Measurement errors
        resid_X = X_proc - F_init @ Lambda_init.T
        sigma_e_init = np.var(resid_X, axis=0)
        sigma_e_init = np.maximum(sigma_e_init, 0.01)  # Lower bound
        
        # Construct state-space matrices
        self.Z = np.hstack([Lambda_init] + [np.zeros((N, self.r))] * (self.p - 1))
        
        # Companion form transition matrix
        self.T_mat = np.zeros((self.m, self.m))
        self.T_mat[:self.r, :] = np.hstack(Phi_list)
        if self.p > 1:
            self.T_mat[self.r:, :self.m - self.r] = np.eye(self.m - self.r)
        
        # State innovation selector
        self.R = np.zeros((self.m, self.r))
        self.R[:self.r, :] = np.eye(self.r)
        
        self.Q = Q_init
        self.H = np.diag(sigma_e_init)
        
        return self
    
    def _enforce_identification(self, Lambda):
        """
        Enforce lower triangular structure on first r x r block.
        """
        L = Lambda.copy()
        for i in range(min(self.r, Lambda.shape[0])):
            for j in range(i + 1, self.r):
                if i < Lambda.shape[0]:
                    L[i, j] = 0
            # Positive diagonal
            if i < Lambda.shape[0] and L[i, i] < 0:
                L[:, i] *= -1
        return L
    
    def e_step(self, X):
        """
        E-step: Compute smoothed states and covariances.
        """
        T_obs = X.shape[0]
        
        # Initial state distribution (unconditional)
        alpha_0 = np.zeros(self.m)
        
        # Solve for stationary covariance: P = T P T' + R Q R'
        try:
            I_m2 = np.eye(self.m**2)
            T_kron = np.kron(self.T_mat, self.T_mat)
            vec_RQR = (self.R @ self.Q @ self.R.T).ravel()
            vec_P0 = solve(I_m2 - T_kron, vec_RQR)
            P_0 = vec_P0.reshape(self.m, self.m)
            P_0 = (P_0 + P_0.T) / 2
        except:
            # Fallback: large variance
            P_0 = np.eye(self.m) * 1e3
        
        # Kalman filter
        alpha_pred, alpha_filt, P_pred, P_filt, loglik = kalman_filter(
            X, self.Z, self.H, self.T_mat, self.R, self.Q, alpha_0, P_0
        )
        
        # Kalman smoother
        alpha_smooth, P_smooth, P_smooth_lag = kalman_smoother(
            alpha_pred, alpha_filt, P_pred, P_filt, self.T_mat
        )
        
        return alpha_smooth, P_smooth, P_smooth_lag, loglik
    
    def m_step(self, X, alpha_smooth, P_smooth, P_smooth_lag):
        """
        M-step: Update parameters given sufficient statistics.
        """
        T_obs, N = X.shape
        
        # Sufficient statistics
        S_11 = np.sum([np.outer(alpha_smooth[t], alpha_smooth[t]) + P_smooth[t]
                       for t in range(T_obs)], axis=0)
        
        S_10 = np.sum([np.outer(alpha_smooth[t], alpha_smooth[t-1]) + P_smooth_lag[t]
                       for t in range(1, T_obs)], axis=0)
        
        S_00 = np.sum([np.outer(alpha_smooth[t], alpha_smooth[t]) + P_smooth[t]
                       for t in range(T_obs - 1)], axis=0)
        
        X_alpha = np.sum([np.outer(X[t], alpha_smooth[t]) for t in range(T_obs)], axis=0)
        
        # Update Z (loadings)
        try:
            Z_new = X_alpha @ solve(S_11, np.eye(self.m))
            Lambda_new = Z_new[:, :self.r]
            Lambda_new = self._enforce_identification(Lambda_new)
            self.Z = np.hstack([Lambda_new] + [np.zeros((N, self.r))] * (self.p - 1))
        except:
            pass  # Keep current Z if update fails
        
        # Update T (transition)
        try:
            self.T_mat = S_10.T @ solve(S_00, np.eye(self.m))
        except:
            pass
        
        # Update Q (factor innovation variance)
        try:
            RQR_new = (S_11[:self.r, :self.r] - self.T_mat[:self.r, :] @ S_10[:, :self.r]) / (T_obs - 1)
            RQR_new = (RQR_new + RQR_new.T) / 2
            self.Q = np.maximum(RQR_new, 1e-6 * np.eye(self.r))  # Ensure positive definite
        except:
            pass
        
        # Update H (measurement error variance - diagonal)
        H_diag = np.zeros(N)
        for i in range(N):
            resid_var = np.sum([
                X[t, i]**2 - 2 * X[t, i] * self.Z[i] @ alpha_smooth[t] +
                self.Z[i] @ (np.outer(alpha_smooth[t], alpha_smooth[t]) + P_smooth[t]) @ self.Z[i]
                for t in range(T_obs)
            ])
            H_diag[i] = max(resid_var / T_obs, 1e-6)  # Lower bound
        
        self.H = np.diag(H_diag)
    
    def fit(self, X, max_iter=100, tol=1e-6, verbose=True):
        """
        Fit model via EM algorithm.
        
        Parameters
        ----------
        X : ndarray (T, N)
            Observed data (demeaned)
        max_iter : int
            Maximum iterations
        tol : float
            Convergence tolerance
        verbose : bool
            Print progress
            
        Returns
        -------
        self
        """
        # Initialize
        if verbose:
            print("Initializing from PCA...")
        self.initialize_from_pca(X)
        
        # EM iterations
        if verbose:
            print(f"\nStarting EM algorithm (max {max_iter} iterations)...\n")
        
        for iteration in range(max_iter):
            # E-step
            alpha_smooth, P_smooth, P_smooth_lag, loglik = self.e_step(X)
            self.loglik_hist_.append(loglik)
            
            if verbose and iteration % 10 == 0:
                print(f"Iteration {iteration:3d}: Log-likelihood = {loglik:.2f}")
            
            # Check convergence
            if iteration > 0:
                ll_change = abs(self.loglik_hist_[-1] - self.loglik_hist_[-2])
                rel_change = ll_change / max(1, abs(self.loglik_hist_[-2]))
                
                if rel_change < tol:
                    if verbose:
                        print(f"\nConverged at iteration {iteration}!")
                    break
            
            # M-step
            self.m_step(X, alpha_smooth, P_smooth, P_smooth_lag)
        
        self.n_iter_ = iteration + 1
        self.loglik_ = self.loglik_hist_[-1]
        
        # Extract parameters
        self.Lambda_ = self.Z[:, :self.r]
        self.Phi_ = self.T_mat[:self.r, :self.r * self.p]
        self.sigma_e_ = np.sqrt(np.diag(self.H))
        
        # Compute information criteria
        n_params = self.r * X.shape[1] + self.r**2 * self.p + self.r*(self.r+1)//2 + X.shape[1]
        T_obs = X.shape[0]
        self.aic_ = -2 * self.loglik_ + 2 * n_params
        self.bic_ = -2 * self.loglik_ + n_params * np.log(T_obs)
        
        if verbose:
            print(f"\nFinal log-likelihood: {self.loglik_:.2f}")
            print(f"AIC: {self.aic_:.2f}")
            print(f"BIC: {self.bic_:.2f}")
        
        return self
    
    def smooth_factors(self, X):
        """Extract smoothed factors."""
        alpha_smooth, P_smooth, _, _ = self.e_step(X)
        F_smooth = alpha_smooth[:, :self.r]
        return F_smooth, alpha_smooth, P_smooth


print("EM algorithm implemented!")

## 4. Demonstration: Simulated Data

Let's test the EM algorithm on simulated data where we know the truth.

In [ ]:
# Purpose: Generate DFM data for testing

def simulate_dfm(T=200, N=30, r=2, p=1, seed=42):
    """Simulate DFM with known parameters."""
    np.random.seed(seed)
    
    # True parameters
    Lambda_true = np.random.randn(N, r) * 0.8
    Phi_true = np.array([[0.7, 0.1], [0.1, 0.6]])
    sigma_e_true = np.ones(N) * 0.5
    
    # Simulate factors
    F = np.zeros((T, r))
    for t in range(1, T):
        F[t] = Phi_true @ F[t-1] + np.random.randn(r) * 0.5
    
    # Simulate observations
    X = F @ Lambda_true.T + np.random.randn(T, N) * sigma_e_true
    X = X - X.mean(axis=0)  # Demean
    
    return X, F, Lambda_true, Phi_true


# Generate data
print("Simulating DFM data...\n")
X, F_true, Lambda_true, Phi_true = simulate_dfm(T=200, N=30, r=2, p=1, seed=42)

print(f"Data dimensions: T={X.shape[0]}, N={X.shape[1]}")
print(f"True parameters:")
print(f"  Number of factors: r={F_true.shape[1]}")
print(f"  VAR order: p=1")
print(f"\nTrue Phi matrix:")
print(Phi_true)

### Fit EM Algorithm

In [ ]:
# Purpose: Fit EM algorithm and monitor convergence

model_em = EMDynamicFactorModel(n_factors=2, n_lags=1)
model_em.fit(X, max_iter=100, tol=1e-6, verbose=True)

### Visualize Convergence

In [ ]:
# Purpose: Plot log-likelihood convergence

fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(model_em.loglik_hist_, linewidth=2.5, marker='o', markersize=4)
ax.set_xlabel('EM Iteration', fontsize=12)
ax.set_ylabel('Log-Likelihood', fontsize=12)
ax.set_title('EM Algorithm Convergence', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nLog-likelihood increased from {model_em.loglik_hist_[0]:.2f} to {model_em.loglik_hist_[-1]:.2f}")
print(f"Total increase: {model_em.loglik_hist_[-1] - model_em.loglik_hist_[0]:.2f}")

### Compare Estimates to Truth

In [ ]:
# Purpose: Evaluate estimation accuracy

print("="*60)
print("PARAMETER RECOVERY")
print("="*60)

print("\nEstimated Phi (first r x r block):")
Phi_est = model_em.Phi_[:, :2]
print(Phi_est)

print("\nTrue Phi:")
print(Phi_true)

print("\nElement-wise errors:")
print(np.abs(Phi_est - Phi_true))

# Extract smoothed factors
F_em, _, _ = model_em.smooth_factors(X)

# Align to true factors
from scipy.linalg import orthogonal_procrustes
H, _ = orthogonal_procrustes(F_em, F_true)
F_aligned = F_em @ H

# Compute correlations
corrs = [np.corrcoef(F_true[:, i], F_aligned[:, i])[0, 1] for i in range(2)]
print("\nFactor recovery (correlation after alignment):")
for i, corr in enumerate(corrs):
    print(f"  Factor {i+1}: {corr:.4f}")

print("="*60)

## Exercise 4.1: Compare EM to PCA

Implement PCA-based estimation and compare the factor estimates to EM.

**Task:** Extract factors using PCA (from Module 3) and compare to EM factors.

In [ ]:
# YOUR CODE HERE
# ---------------

def compare_em_vs_pca(X, model_em):
    """
    Compare EM and PCA factor estimates.
    
    Parameters
    ----------
    X : ndarray (T, N)
        Data
    model_em : EMDynamicFactorModel
        Fitted EM model
        
    Returns
    -------
    comparison_stats : dict
        Statistics comparing the two methods
    """
    # TODO: Extract factors using PCA
    # Hint: Use sklearn.decomposition.PCA or implement from scratch
    F_pca = None  # Replace this
    
    # TODO: Get EM factors
    F_em, _, _ = model_em.smooth_factors(X)
    
    # TODO: Align factors (PCA and EM have arbitrary rotations)
    # Use orthogonal_procrustes
    
    # TODO: Compute correlation between aligned factors
    
    # TODO: Create visualization comparing PCA vs EM factors
    
    comparison_stats = None  # Replace with your statistics
    
    return comparison_stats


# Test your implementation
# stats = compare_em_vs_pca(X, model_em)

<details>
<summary>Hint 1</summary>
Use PCA with n_components=2. Remember to standardize X first.
</details>

<details>
<summary>Hint 2</summary>
Use orthogonal_procrustes(F_pca, F_em) to align the factors before comparison.
</details>

In [ ]:
# AUTO-GRADED TESTS
# ------------------

def test_exercise_4_1():
    """Test EM vs PCA comparison."""
    stats = compare_em_vs_pca(X, model_em)
    
    assert stats is not None, "Don't forget to return comparison_stats!"
    assert 'correlations' in stats, "Should include factor correlations"
    
    corrs = stats['correlations']
    assert len(corrs) == 2, "Should have 2 correlations (one per factor)"
    assert all(abs(c) > 0.8 for c in corrs), "Factors should be highly correlated"
    
    print("✅ Exercise 4.1 passed!")
    print(f"\nPCA-EM factor correlations: {corrs}")
    print(f"Mean correlation: {np.mean(np.abs(corrs)):.3f}")

# Uncomment to test:
# test_exercise_4_1()

### Solution to Exercise 4.1

In [ ]:
# SOLUTION
# --------

def compare_em_vs_pca(X, model_em):
    """
    Compare EM and PCA factor estimates.
    """
    from sklearn.decomposition import PCA
    from sklearn.preprocessing import StandardScaler
    
    # Standardize and apply PCA
    scaler = StandardScaler()
    X_std = scaler.fit_transform(X)
    
    pca = PCA(n_components=2)
    F_pca = pca.fit_transform(X_std)
    
    # Get EM factors
    F_em, _, _ = model_em.smooth_factors(X)
    
    # Align factors
    H, _ = orthogonal_procrustes(F_pca, F_em)
    F_pca_aligned = F_pca @ H
    
    # Compute correlations
    correlations = [np.corrcoef(F_em[:, i], F_pca_aligned[:, i])[0, 1] for i in range(2)]
    
    # Visualization
    fig, axes = plt.subplots(2, 1, figsize=(12, 8))
    
    for i in range(2):
        ax = axes[i]
        ax.plot(F_pca_aligned[:, i], label='PCA', alpha=0.7, linewidth=2)
        ax.plot(F_em[:, i], label='EM', alpha=0.7, linewidth=2, linestyle='--')
        ax.set_title(f'Factor {i+1} (correlation: {correlations[i]:.3f})',
                    fontsize=12, fontweight='bold')
        ax.set_xlabel('Time')
        ax.set_ylabel('Factor Value')
        ax.legend()
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Compute MSE
    mse = np.mean((F_pca_aligned - F_em)**2, axis=0)
    
    comparison_stats = {
        'correlations': correlations,
        'mse': mse,
        'mean_correlation': np.mean(np.abs(correlations)),
        'variance_explained_pca': pca.explained_variance_ratio_
    }
    
    print("\n" + "="*60)
    print("PCA vs EM COMPARISON")
    print("="*60)
    print(f"\nFactor correlations:")
    for i, corr in enumerate(correlations):
        print(f"  Factor {i+1}: {corr:.4f}")
    print(f"\nMean absolute correlation: {comparison_stats['mean_correlation']:.4f}")
    print(f"\nMSE (aligned):")
    for i, err in enumerate(mse):
        print(f"  Factor {i+1}: {err:.4f}")
    print("\nConclusion: EM and PCA produce very similar factor estimates!")
    print("="*60)
    
    return comparison_stats


# Run comparison
stats = compare_em_vs_pca(X, model_em)

## Summary

### Key Takeaways

1. **EM is guaranteed to increase likelihood:** Each iteration improves (or maintains) the log-likelihood

2. **E-step = Kalman smoother:** Computing expected sufficient statistics requires full-sample smoothed estimates

3. **M-step has closed-form updates:** Given smoothed states, parameter updates are just regressions

4. **PCA initialization is crucial:** Starting from PCA puts EM close to the optimum

5. **EM ≈ PCA asymptotically:** For large samples, EM and PCA give very similar results

6. **EM advantages:**
   - Better small-sample properties
   - Natural uncertainty quantification
   - Handles missing data elegantly
   - Allows complex constraints

7. **EM disadvantages:**
   - Slower than PCA
   - Requires good initialization
   - Can converge slowly

### When to Use EM

- **Small samples** (T < 100): EM is more efficient
- **Missing data**: EM naturally handles irregular patterns
- **Need standard errors**: Information matrix provides uncertainty
- **Complex constraints**: Cross-equation restrictions, sign constraints

### When to Use PCA

- **Large samples** (T > 200): PCA is faster with similar accuracy
- **Quick exploration**: No convergence monitoring needed
- **Initialization**: Use PCA to start EM

### What's Next

In the next notebook, we'll explore **Bayesian estimation** using MCMC methods (Gibbs sampling), which provides full posterior distributions for parameters and handles uncertainty in a principled way.

### Additional Resources

- Shumway & Stoffer (1982): "An Approach to Time Series Smoothing and Forecasting Using the EM Algorithm"
- Durbin & Koopman (2012): *Time Series Analysis by State Space Methods*, Chapter 7
- Module guide: `guides/02_em_algorithm_dfm.md`

---

**Completion:** Module 4, Notebook 1 complete! Proceed to Notebook 2 for Bayesian methods.